# Chess Move Error Detection

## Objective

The goal of this project is to identify an erroneous move in a chess game.

Each example contains the first **40 plies** of a real chess game, written in **SAN (Standard Algebraic Notation)**.

Exactly one of the first **10 plies** has been replaced by a move taken from the **same ply position in another real game**.

For example:

```text
Original:
e4 e5 Nf3 Nc6 Bb5 a6 Ba4 Nf6 O-O Be7 ...

Corrupted:
e4 e5 Nf3 e6 Bb5 a6 Ba4 Nf6 O-O Be7 ...
         ^^^^^
```

In this example the corrupted move is at position **4**, so the target is `4`.

The task is therefore a **10-class classification problem**: for each corrupted sequence, predict which one of positions **1,...,10** contains the error.

The remaining moves, up to ply 40, are part of the input and may contain useful information for identifying an earlier corruption.


## Important constraints

The objective of the project is to learn exclusively from the games provided in the dataset.

Your solution must not use, either explicitly or implicitly, any external knowledge about chess. In particular, you should assume that the rules of the game are unknown and must not be encoded in your model or preprocessing.

Pretrained chess models, chess engines, opening books, move generators, legality checks, or other sources of chess-specific knowledge are not allowed.

The total number of trainable parameters of your model must not exceed **6 million**.


## Download the datasets

In [1]:
!pip -q install gdown

In [2]:
import os
import gdown

# Replace these IDs with the Google Drive file IDs supplied with the project.
TRAIN_FILE_ID = "1xyggntfZ2-6BTAagxJm-tFKDXerTAs8c"
TEST_FILE_ID  = "1VozRpr3dlVpA17BL-7ALCaOXa2vCe64J"

TRAIN_PATH = "chess_error_detection_train.csv"
TEST_PATH  = "chess_error_detection_test.csv"

if not os.path.exists(TRAIN_PATH):
    gdown.download(id=TRAIN_FILE_ID, output=TRAIN_PATH, quiet=False)

if not os.path.exists(TEST_PATH):
    gdown.download(id=TEST_FILE_ID, output=TEST_PATH, quiet=False)

Downloading...
From: https://drive.google.com/uc?id=1xyggntfZ2-6BTAagxJm-tFKDXerTAs8c
To: /content/chess_error_detection_train.csv
100%|██████████| 66.6M/66.6M [00:00<00:00, 94.4MB/s]
Downloading...
From: https://drive.google.com/uc?id=1VozRpr3dlVpA17BL-7ALCaOXa2vCe64J
To: /content/chess_error_detection_test.csv
100%|██████████| 8.33M/8.33M [00:00<00:00, 35.0MB/s]


The public data consist of:

- **400,000 training examples**
- **50,000 test examples**

You are free to use part of training data for validation, but keep test for the final assessment.


## Load and inspect the data

In [3]:
import pandas as pd
import numpy as np

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train shape:", train_df.shape)
print("Test shape: ", test_df.shape)

display(train_df.head())


Train shape: (400000, 3)
Test shape:  (50000, 3)


,sequence,error_position,correct_move
0,e4 c6 f4 Nf6 Nc3 g6 Nf3 Bg7 d4 O-O h3 c5 d5 Nb...,2,d6
1,d4 Nf6 c4 exf4 Nc3 Bb4 Bd2 Nc6 e3 O-O Nf3 b6 d...,4,e6
2,e3 g6 d4 Bg7 c4 d6 Nf3 Nf6 Nf3 O-O O-O Nbd7 Nc...,9,Bd3
3,e4 b6 Nf3 Bb7 d4 d5 e5 e6 Bd3 Qe7 O-O fxe5 Qe1...,10,f6
4,d4 Nf6 b3 g6 Nc3 Bg7 e4 d6 f3 e5 dxe5 dxe5 Qxd...,3,c4


Each row contains three columns:

- `sequence`: the corrupted sequence of 40 SAN moves;
- `error_position`: the position of the corrupted move, from **1 to 10**;
- `correct_move`: the original move that was replaced.

For the present task, the prediction target is **only `error_position`**: you are not supposed to guess the correct move.

The `correct_move` column is provided for completeness and possible analysis, but it must not be used as an input feature.


In [ ]:
print(train_df.columns.tolist())

assert set(train_df.columns) == {"sequence", "error_position", "correct_move"}
assert set(test_df.columns) == {"sequence", "error_position", "correct_move"}

train_lengths = train_df["sequence"].str.split().str.len()
test_lengths = test_df["sequence"].str.split().str.len()

print("Train sequence lengths:")
print(train_lengths.value_counts().sort_index())

print("\nTest sequence lengths:")
print(test_lengths.value_counts().sort_index())

assert (train_lengths == 40).all()
assert (test_lengths == 40).all()
assert train_df["error_position"].between(1, 10).all()
assert test_df["error_position"].between(1, 10).all()


['sequence', 'error_position', 'correct_move']
Train sequence lengths:
sequence
40    400000
Name: count, dtype: int64

Test sequence lengths:
sequence
40    50000
Name: count, dtype: int64


## Inspect the class distribution

In [ ]:
train_distribution = train_df["error_position"].value_counts(normalize=True).sort_index()
test_distribution = test_df["error_position"].value_counts(normalize=True).sort_index()

distribution = pd.DataFrame({
    "train": train_distribution,
    "test": test_distribution,
})

display(distribution)


,train,test
error_position,,
1,0.099727,0.09972
2,0.100695,0.10070
3,0.100165,0.10018
4,0.099255,0.09926
5,0.100720,0.10072
6,0.099965,0.09996
7,0.099625,0.09962
8,0.099903,0.09990
9,0.099748,0.09974


The corruption position is sampled uniformly among the first 10 plies.

Therefore, a random classifier has an expected Accuracy@1 of approximately **10%**.


## Inspect individual examples

In [ ]:
sample = train_df.sample(1)

for _, row in sample.iterrows():
    moves = row["sequence"].split()
    pos = int(row["error_position"])

    print("Sequence:")
    for i, move in enumerate(moves, start=1):
        marker = "  <-- corrupted" if i == pos else ""
        print(f"{i:2d}: {move}{marker}")

    print("Correct move:", row["correct_move"])
    print("-" * 60)


Sequence:
 1: e4
 2: e5
 3: Nf3
 4: Nc6
 5: Bb5
 6: d6
 7: Bxc6+
 8: bxc6
 9: Nc3  <-- corrupted
10: exd4
11: Nxd4
12: c5
13: Nf3
14: Bg4
15: Qd3
16: Rb8
17: O-O
18: h6
19: h3
20: Bh5
21: Nh2
22: Be7
23: Qg3
24: Bf6
25: Re1
26: Ne7
27: c3
28: Nc6
29: Nd2
30: Ne5
31: f4
32: Nc6
33: Nc4
34: O-O
35: e5
36: Bh4
37: Qe3
38: Bxe1
39: Qxe1
40: d5
Correct move: d4
------------------------------------------------------------


## Expected model output

For each input sequence, your model should produce a probability distribution over the 10 possible error positions:

$$
(p_1,p_2,\ldots,p_{10}),
\qquad
\sum_{i=1}^{10} p_i = 1.
$$

For example:

```text
[0.01, 0.03, 0.08, 0.72, 0.05, 0.03, 0.02, 0.02, 0.02, 0.02]
```

The predicted error position is the position with maximum probability:

```text
prediction = 4
```

Internally, many libraries use class labels `0,...,9`. This is perfectly fine, but remember that the dataset uses the human-readable convention **1,...,10**.


Evaluation metric: Accuracy

The official metric is **Accuracy@1**, i.e. ordinary top-1 classification accuracy.

A prediction is counted as correct only when the position with highest predicted probability is exactly the true corrupted position.


In [ ]:
def accuracy_at_1(y_true, probabilities):
    y_true = np.asarray(y_true)
    probabilities = np.asarray(probabilities)

    assert probabilities.ndim == 2
    assert probabilities.shape[1] == 10
    assert len(y_true) == len(probabilities)

    # argmax returns classes 0,...,9, hence +1.
    y_pred = np.argmax(probabilities, axis=1) + 1

    return np.mean(y_pred == y_true)


### Example

In [ ]:
y_true = np.array([4, 2, 7])

probabilities = np.array([
    [0.01, 0.03, 0.08, 0.72, 0.05, 0.03, 0.02, 0.02, 0.02, 0.02],
    [0.10, 0.50, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05],
    [0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.40, 0.10, 0.10, 0.10],
])

print("Accuracy@1:", accuracy_at_1(y_true, probabilities))


## Final evaluation

You are supposed to evaluate your model on the full test set.

Your submitted notebook should clearly report:

- the architecture of the final model and its ratio;
- the total number of trainable parameters;
- the Accuracy@1 obtained on the public test set.

## Good Work!